# Taller 6: Anotación Estructural y Funcional de Genomas
## Descifrando la Maquinaria Degradadora de Plásticos en *Ideonella sakaiensis*

**Pontificia Universidad Javeriana**  
**Facultad de Ingeniería — Departamento de Procesos**  
**Bioinformática para Ingenieros**  

* **Docente:** Msc. Juan Sebastián Malagón Torres  
* **Asistente de docencia:** Eliana Zuluaga Aponte  

---

### Objetivos de Aprendizaje
1. **Comprender los principios de la anotación genómica estructural y funcional:** Diferenciar entre la determinación de coordenadas génicas/ORFs (*ab initio* y evidencias) y la asignación ontológica de funciones biológicas a los productos génicos predichos.
2. **Ejecutar un pipeline integral de anotación procariota con Prokka:** Coordinar la identificación rápida de marcos abiertos de lectura (CDS con Prodigal), RNAs de transferencia (tRNA con Aragorn), RNAs ribosomales (rRNA con Barrnap) y repeticiones CRISPR en el genoma ensamblado de *Ideonella sakaiensis*.
3. **Interpretar la anotación funcional con eggNOG-mapper y Gene Ontology (GO):** Utilizar grupos de ortología evolutiva para transferir anotaciones funcionales precisas (términos GO, vías metabólicas KEGG, códigos COG y enzimas CAZy), evitando la propagación de errores por parálogos.
4. **Descubrir la maquinaria de degradación y asimilación de plásticos (PET):** Minar el proteoma predicho para caracterizar la isla catabólica del PET, identificando la **PETasa** (*poly(ethylene terephthalate) hydrolase*, EC 3.1.1.101), la **MHETasa** (*mono(2-hydroxyethyl) terephthalate hydrolase*, EC 3.1.1.102) y el operón dioxigenasa del ácido tereftálico (*tphA1, tphA2, tphA3, tphB*).
5. **Evaluar rigurosamente la calidad biológica con BUSCO:** Medir la representatividad de genes ortólogos universales de copia única (*Single-Copy Orthologs*) sobre el proteoma predicho con el linaje `bacteria_odb10`.
6. **Diseñar visualizaciones científicas en Python:** Generar gráficos de distribución de longitudes de CDS, mapas del clúster génico de degradación de PET, términos GO enriquecidos y barras apiladas de completitud BUSCO.

> [!NOTE]
> **Dinámica de la sesión (2 a 3 horas):**  
> * **Parte 1 (Yo Hago — Demostración Guiada):** Inspeccionamos el genoma de *Ideonella sakaiensis* 201-F6 y ejecutamos la anotación estructural automatizada con **Prokka**, analizando sus métricas clave.  
> * **Parte 2 (Tú Haces — Práctica Autónoma):** Cargamos y exploramos las anotaciones funcionales de **eggNOG-mapper** en Python, desglosando ontologías moleculares (GO terms) y rutas KEGG.  
> * **Parte 3 (Minería Bioinformática):** Rastrearemos los genes responsables de la degradación del plástico PET, descubriendo cómo están agrupados físicamente en el genoma bacteriano.  
> * **Parte 4 (Evaluamos Juntos — Calidad Biológica):** Evaluamos el proteoma con **BUSCO** (`bacteria_odb10`) y analizamos el impacto de la completitud en proyectos de ingeniería metabólica.


## 0. Preparación del Entorno y Verificación de Herramientas

¡Hola bioniños! Es un gusto tenerlos de regreso en nuestro laboratorio computacional. Hoy nos sumergiremos en uno de los problemas biológicos y ambientales más apasionantes de la última década: la biodegradación de polímeros sintéticos.

Para desarrollar este taller utilizaremos nuestro entorno de Conda `bioinformatics`, el cual cuenta con los motores de anotación y análisis:
* **Prokka (`prokka`):** Pipeline rápido de anotación estructural y funcional primaria para genomas procariotas.
* **BUSCO (`busco`):** Evaluación de la completitud biológica mediante genes ortólogos universales.
* **Librerías de análisis y bioinformática en Python:** BioPython, Pandas, NumPy, Matplotlib y Seaborn.

Ejecuta la siguiente celda para inicializar las librerías, configurar la estética gráfica y verificar la disponibilidad de los programas:

In [ ]:
import os
import sys
import subprocess
import glob
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO

# Configuración del estilo visual institucional
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"
plt.rcParams["axes.edgecolor"] = "#cccccc"
plt.rcParams["axes.linewidth"] = 0.8

# Aseguramos que el PATH priorice el entorno de bioinformática
os.environ["PATH"] = f"/home/jmalagont/miniconda3/envs/bioinformatics/bin:{os.environ.get('PATH', '')}"

print("=" * 60)
print("VERIFICACIÓN DE HERRAMIENTAS Y ENTORNO BIOINFORMÁTICO")
print("=" * 60)

for cmd in ["prokka --version", "busco --version"]:
    res = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    linea = (res.stdout or res.stderr).strip().split('\n')[0]
    print(f" [OK] {linea}")

print("=" * 60)
print("¡Entorno verificado y listo para la anotación!")

### 0.1 Inspección del Genoma de Entrada: *Ideonella sakaiensis* 201-F6

En 2016, un equipo liderado por Shosuke Yoshida descubrió en los sedimentos de una planta de reciclaje de botellas plásticas en Sakai (Japón) una bacteria gramnegativa de la clase Betaproteobacteria capaz de utilizar el plástico **PET** (tereftalato de polietileno) como su principal fuente de carbono y energía para subsistir (*Science*, 2016).

En la carpeta `data/` disponemos del archivo ensamblado `Ideonella-sakaiensis.fna`. Vamos a examinar cuántos replicones componen este genoma, su tamaño total y su contenido de GC:

In [ ]:
genome_file = "data/Ideonella-sakaiensis.fna" if os.path.exists("data/Ideonella-sakaiensis.fna") else "Ideonella-sakaiensis.fna"

secuencias = []
longitudes = []
gcs = []

for record in SeqIO.parse(genome_file, "fasta"):
    secuencias.append(record.id)
    seq_len = len(record.seq)
    longitudes.append(seq_len)
    gc_val = 100.0 * (record.seq.count("G") + record.seq.count("C")) / seq_len
    gcs.append(gc_val)
    print(f"🔹 {record.id}: Longitud = {seq_len:,} bp | %GC = {gc_val:.2f}% | Descripción = {record.description}")

total_bp = sum(longitudes)
gc_global = sum([rec.seq.count("G") + rec.seq.count("C") for rec in SeqIO.parse(genome_file, "fasta")]) * 100.0 / total_bp

print("-" * 60)
print(f"Total replicones (cromosoma + plásmido): {len(secuencias)}")
print(f"Tamaño total del genoma: {total_bp:,} bp ({total_bp / 1e6:.2f} Mb)")
print(f"Contenido GC global: {gc_global:.2f}%")

## 1. Demostración Guiada ("Yo Hago"): Anotación Estructural con Prokka

La **anotación estructural** es el proceso de identificar y delimitar físicamente las coordenadas de todos los elementos biológicamente funcionales en la secuencia de nucleótidos: genes codificantes de proteínas (CDS), señales de inicio/parada, secuencias Shine-Dalgarno, RNAs de transferencia (tRNAs), RNAs ribosomales (rRNAs), repeticiones CRISPR y regiones regulatorias.

A diferencia de los eucariotas (que presentan intrones, exones y splicing alternativo), las bacterias poseen una **alta densidad codificante (~85-90%)** y operones policistrónicos continuos.

**Prokka** es una suite bioinformática altamente eficiente que orquesta en paralelo múltiples algoritmos especializados:
1. **Prodigal:** Predicción *ab initio* de marcos abiertos de lectura y regiones codificantes (CDS).
2. **Aragorn:** Detección de RNAs de transferencia (tRNA) y tmRNA.
3. **Barrnap:** Detección de RNAs ribosomales (5S, 16S, 23S rRNA).
4. **MinCED:** Detección de repeticiones CRISPR.
5. **Búsqueda jerárquica con BLAST+/HMMER:** Compara las proteínas predichas contra bases de datos curadas (IS, AMR, SwissProt, HAMAP) para asignar nombres de genes y productos.

Vamos a ejecutar Prokka sobre el genoma de *Ideonella sakaiensis*:

In [ ]:
# =========================================================================
# PARÁMETROS CONFIGURABLES: Anotación con Prokka
# =========================================================================
GENOMA = "data/Ideonella-sakaiensis.fna" if os.path.exists("data/Ideonella-sakaiensis.fna") else "Ideonella-sakaiensis.fna"
OUT_PROKKA = "resultados/prokka_ideonella"
HILOS = 8

# Comando Prokka para anotar bacterias procariotas
cmd_prokka = (
    f"prokka --outdir {OUT_PROKKA} "
    f"--prefix ideonella_sakaiensis "
    f"--genus Ideonella --species sakaiensis --strain 201-F6 "
    f"--kingdom Bacteria --cpus {HILOS} --force {GENOMA}"
)

print(f"Ejecutando Prokka:\n{cmd_prokka}\n")

if not os.path.exists(f"{OUT_PROKKA}/ideonella_sakaiensis.tsv"):
    res = subprocess.run(cmd_prokka, shell=True, capture_output=True, text=True)
    lineas_salida = (res.stdout or res.stderr).strip().split("\n")
    print("\n".join(lineas_salida[-15:]))
else:
    print(" [INFO] La anotación de Prokka ya existe en resultados/prokka_ideonella. ¡Cargando resultados!")

### 1.2 Análisis Cuantitativo de los Resultados Estructurales

Prokka genera una amplia colección de archivos en `resultados/prokka_ideonella/`:
* `.txt`: Estadísticas globales de las características encontradas.
* `.tsv`: Tabla con todos los features anotados (locus tag, tipo, longitud, gen, producto).
* `.gff`: Anotación completa en formato estándar GFF3 con coordenadas y secuencias.
* `.faa`: Secuencias traducidas de aminoácidos de todas las proteínas predichas (CDS).
* `.ffn`: Secuencias de nucleótidos de todos los genes predichos.
* `.gbk`: Archivo enriquecido en formato GenBank compatible con visualizadores como Artemis o Mauve.

Analicemos cuantitativamente la proporción de genes codificantes, la densidad codificante del genoma y cuántas proteínas quedaron etiquetadas como *hypothetical protein*:

In [ ]:
prokka_txt = f"{OUT_PROKKA}/ideonella_sakaiensis.txt"
with open(prokka_txt) as f:
    stats_text = f.read()

print("=" * 50)
print("ESTADÍSTICAS REPORTADAS POR PROKKA (.txt)")
print("=" * 50)
print(stats_text)

# Lectura de la tabla TSV generada por Prokka
prokka_tsv = f"{OUT_PROKKA}/ideonella_sakaiensis.tsv"
df_prokka = pd.read_csv(prokka_tsv, sep="\t")

cds_total = len(df_prokka[df_prokka["ftype"] == "CDS"])
hipoteticas = len(df_prokka[(df_prokka["ftype"] == "CDS") & (df_prokka["product"] == "hypothetical protein")])
con_funcion = cds_total - hipoteticas
pct_funcion = (con_funcion / cds_total) * 100.0

# Cálculo de la densidad codificante
longitud_cds_total = df_prokka[df_prokka["ftype"] == "CDS"]["length_bp"].sum()
densidad_codificante = (longitud_cds_total / total_bp) * 100.0

print("=" * 50)
print(f"Total de CDS predichos: {cds_total}")
print(f"Proteínas con función asignada por Prokka: {con_funcion} ({pct_funcion:.2f}%)")
print(f"Proteínas hipotéticas (sin función en Prokka): {hipoteticas} ({100 - pct_funcion:.2f}%)")
print(f"Densidad codificante del genoma: {densidad_codificante:.2f}%")
print("=" * 50)

### 1.3 Gráfico 1: Distribución de Longitudes de Secuencias Codificantes (CDS)

En bacterias, el tamaño típico de una proteína oscila entre 250 y 400 aminoácidos (~750 a 1,200 nucleótidos). Un ensamblaje o anotación con errores sistemáticos (por ejemplo, corrimientos de pauta de lectura / *frameshifts*) suele mostrar picos anómalos de proteínas excesivamente cortas (<60 aa) debido a la aparición espuria de codones de parada prematuros.

Grafiquemos la distribución de longitud de las proteínas predichas en *I. sakaiensis*:

In [ ]:
plt.figure(figsize=(10, 5))

# Extraer longitudes en aminoácidos (longitud en bp dividido entre 3)
longitudes_aa = df_prokka[df_prokka["ftype"] == "CDS"]["length_bp"] / 3.0

sns.histplot(longitudes_aa, bins=60, kde=True, color="#2b5c8f", edgecolor="white", alpha=0.7)

mediana_len = np.median(longitudes_aa)
media_len = np.mean(longitudes_aa)

plt.axvline(mediana_len, color="#d95f02", linestyle="--", linewidth=2, label=f"Mediana: {mediana_len:.0f} aa")
plt.axvline(media_len, color="#7570b3", linestyle=":", linewidth=2, label=f"Media: {media_len:.0f} aa")

plt.title("Distribución de Longitudes de Proteínas (CDS) — Ideonella sakaiensis 201-F6", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Longitud de la proteína (aminoácidos)", fontsize=11)
plt.ylabel("Número de CDS", fontsize=11)
plt.xlim(0, 1500)
plt.legend(frameon=True, facecolor="white", edgecolor="#cccccc")
plt.tight_layout()
plt.show()

## 2. Práctica Autónoma ("Tú Haces"): Anotación Funcional con eggNOG-mapper

La **anotación funcional** busca responder a la pregunta fundamental: *¿qué hacen las proteínas descubiertas y en qué procesos celulares participan?*

**eggNOG-mapper** basa su poder predictivo en la base de datos **eggNOG** (*Evolutionary genealogy of genes: Non-supervised Orthologous Groups*). El concepto clave es la **ortología**:
* **Ortólogos:** Genes en distintas especies que derivaron de un ancestro común mediante especiación; retienen casi con certeza la misma función biológica.
* **Parálogos:** Genes generados por duplicación dentro de un mismo genoma; frecuentemente divergen y adquieren nuevas funciones.

Alinear simplemente contra NR (NCBI) con BLAST tradicional propaga anotaciones erróneas de parálogos. eggNOG-mapper utiliza algoritmos ultrarrápidos (como DIAMOND) contra grupos precalculados de ortología, transfiriendo una batería completa de información ontológica:
* **Gene Ontology (GO):** Proceso Biológico (BP), Función Molecular (MF) y Componente Celular (CC).
* **KEGG Pathways:** Rutas metabólicas y números de enzima (EC).
* **COG/NOG:** Categorías funcionales evolutivas.
* **CAZy:** Enzimas activas en carbohidratos.

En la carpeta `data/eggnog/` disponemos del archivo precalculado `ideonella_sakaiensis.emapper.annotations` generado a partir del `.faa` de Prokka. Vamos a cargarlo con Pandas para explorarlo:

In [ ]:
eggnog_file = "data/eggnog/ideonella_sakaiensis.emapper.annotations"

# Leemos la tabla omitiendo las dos primeras líneas de metadatos
df_eggnog = pd.read_csv(eggnog_file, sep="\t", skiprows=2)
df_eggnog.rename(columns={"#query": "query"}, inplace=True)

print("=" * 70)
print(f"REGISTROS DE ANOTACIÓN FUNCIONAL (eggNOG-mapper): {len(df_eggnog)}")
print("=" * 70)
print(f"Columnas disponibles: {df_eggnog.columns.tolist()}\n")
df_eggnog[["query", "Description", "Preferred_name", "GOs", "EC", "COG_category"]].head(5)

## 3. Minería Bioinformática: Descubriendo la Ruta de Degradación de Plásticos (PET)

El tereftalato de polietileno (**PET**) es un poliéster termoplástico ampliamente utilizado en botellas de bebidas y fibras textiles debido a su estabilidad química y resistencia mecánica. Sin embargo, su persistencia en el medio ambiente representa un desafío ecológico monumental.

*Ideonella sakaiensis* desarrolló una vía enzimática de dos etapas secuenciales para hidrolizar el PET a temperatura ambiente:
1. **PETasa (*poly(ethylene terephthalate) hydrolase*, EC 3.1.1.101):**  
   Enzima secretada al medio extracelular (perteneciente a la superfamilia $\alpha/\beta$-hidrolasa, tipo cutinasa/esterasa). Rompe los enlaces éster del PET insoluble de cadena larga, generando principalmente **MHET** (*mono(2-hydroxyethyl) terephthalate*), además de trazas de ácido tereftálico (TPA) y bis(2-hidroxietil) tereftalato (BHET).
2. **MHETasa (*mono(2-hydroxyethyl) terephthalate hydrolase*, EC 3.1.1.102):**  
   Enzima anclada a la membrana externa/periplasma (familia de las tanasas/feruloil esterasas). Hidroliza el MHET soluble en sus monómeros fundamentales: **ácido tereftálico (TPA)** y **etilenglicol (EG)**.
3. **Complejo Tereftalato 1,2-Dioxigenasa (*tphA1, tphA2, tphA3, tphB*):**  
   Importa el TPA al citoplasma y lo convierte en protocatecuato (PCA). Posteriormente, la enzima protocatecuato 3,4-dioxigenasa (*pcaG, pcaH*) rompe el anillo aromático mediante la ruta del $\beta$-cetoadipato, conectando la degradación del plástico con el **ciclo de Krebs (TCA)**.

Vamos a filtrar en nuestro dataset las enzimas clave de esta ruta catabólica:

In [ ]:
# Genes de interés en la asimilación del PET
plastic_genes = ["PETase", "MHETase", "tphA1", "tphA2", "tphA3", "tphB"]

df_plastic = df_eggnog[df_eggnog["Preferred_name"].isin(plastic_genes)].copy()

# Añadimos la longitud del CDS desde la tabla de Prokka
longitudes_cds = []
for q in df_plastic["query"]:
    row_p = df_prokka[df_prokka["locus_tag"] == q].iloc[0]
    longitudes_cds.append(f"{row_p['length_bp']} bp ({row_p['length_bp']//3} aa)")

df_plastic["Longitud_Gen"] = longitudes_cds

cols_tabla = ["query", "Preferred_name", "Description", "EC", "Longitud_Gen", "GOs"]
print("=" * 80)
print("ENZIMAS DE LA RUTA CATABÓLICA DE DEGRADACIÓN DE PET IDENTIFICADAS")
print("=" * 80)
df_plastic[cols_tabla].reset_index(drop=True)

### 3.1 Identificación de Términos Gene Ontology (GO) Asociados a la Degradación de Plásticos

**Gene Ontology (GO)** provee un vocabulario computacional estructurado y universal organizado en tres ontologías ortogonales:
1. **Proceso Biológico (Biological Process - BP):** ¿En qué objetivo o ruta biológica participa la proteína?  
   * `GO:0042178`: *xenobiotic catabolic process* (degradación de compuestos sintéticos/antropogénicos no naturales).
   * `GO:0019439`: *aromatic compound catabolic process* (ruptura de anillos aromáticos como el tereftalato).
2. **Función Molecular (Molecular Function - MF):** ¿Cuál es la actividad bioquímica específica a nivel molecular?  
   * `GO:0052689`: *carboxylic ester hydrolase activity* (hidrólisis de enlaces éster carboxílico presentes en el PET y MHET).
   * `GO:0018598`: *terephthalate 1,2-dioxygenase activity* (incorporación de oxígeno molecular al anillo del TPA).
3. **Componente Celular (Cellular Component - CC):** ¿Dónde se ubica físicamente el producto génico?  
   * `GO:0005576`: *extracellular region* (la PETasa se secreta al exterior para atacar el plástico macizo).
   * `GO:0009279`: *cell outer membrane* (la MHETasa se ancla en la membrana externa para capturar los dímeros solubles).

Examinemos cuáles genes del genoma completo de *I. sakaiensis* están asignados específicamente al proceso catabólico de xenobióticos (`GO:0042178`):

In [ ]:
# Desglosamos todos los GO terms asignados en el genoma
todos_gos = []
for gos in df_eggnog["GOs"].dropna():
    if gos != "-":
        for g in gos.split(","):
            g = g.strip()
            if g.startswith("GO:"):
                todos_gos.append(g)

df_go_counts = pd.Series(todos_gos).value_counts().reset_index()
df_go_counts.columns = ["GO_ID", "Frecuencia"]

# Diccionario biológico de los GO terms observados
go_names = {
    "GO:0005524": "ATP binding (MF)",
    "GO:0003677": "DNA binding (MF)",
    "GO:0016020": "membrane component (CC)",
    "GO:0005737": "cytoplasm (CC)",
    "GO:0008152": "metabolic process (BP)",
    "GO:0006355": "regulation of transcription (BP)",
    "GO:0042178": "xenobiotic catabolic process (BP) ⭐",
    "GO:0052689": "carboxylic ester hydrolase (MF) ⭐",
    "GO:0016787": "hydrolase activity (MF)",
    "GO:0019439": "aromatic compound catabolism (BP) ⭐",
    "GO:0018598": "terephthalate 1,2-dioxygenase (MF) ⭐",
    "GO:0009279": "cell outer membrane (CC)",
    "GO:0005576": "extracellular region (CC)",
    "GO:0003824": "catalytic activity (MF)",
    "GO:0006810": "transport (BP)"
}

df_go_counts["Nombre_Funcional"] = df_go_counts["GO_ID"].map(lambda x: go_names.get(x, "Other process/function"))

# Filtramos específicamente los genes del proceso catabólico de xenobióticos
df_xeno = df_eggnog[df_eggnog["GOs"].astype(str).str.contains("GO:0042178")][["query", "Preferred_name", "Description", "EC", "GOs"]]

print("=" * 70)
print(f"GENES ANOTADOS CON PROCESO CATABÓLICO DE XENOBIÓTICOS (GO:0042178): {len(df_xeno)}")
print("=" * 70)
df_xeno.reset_index(drop=True)

### 3.2 Gráfico 2: Top 15 Términos Gene Ontology (GO) Más Frecuentes

Comparemos la abundancia de términos funcionales basales (como unión a ATP o actividad catalítica general) frente a los términos especializados que dotan a *I. sakaiensis* de su singular capacidad biotecnológica:

In [ ]:
top_gos = df_go_counts.head(15).copy()

# Asignamos color coral/naranja si está asociado a plásticos/xenobióticos, azul para otros
colores_go = ["#d95f02" if "⭐" in str(row) else "#4575b4" for row in top_gos["Nombre_Funcional"]]

plt.figure(figsize=(10, 6))
bars = plt.barh(top_gos["GO_ID"] + " — " + top_gos["Nombre_Funcional"], top_gos["Frecuencia"], color=colores_go, edgecolor="#333333", alpha=0.85)

plt.gca().invert_yaxis()
plt.title("Top 15 Términos Gene Ontology (GO) en Ideonella sakaiensis\n(Naranja = Enriquecimiento en Degradación de Xenobióticos / Plásticos)", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("Frecuencia (Número de genes anotados)", fontsize=11)
plt.ylabel("Término Gene Ontology", fontsize=11)

for bar in bars:
    w = bar.get_width()
    plt.text(w + 5, bar.get_y() + bar.get_height()/2, f"{int(w)}", va="center", fontsize=9, color="#333333")

plt.xlim(0, max(top_gos["Frecuencia"]) * 1.12)
plt.tight_layout()
plt.show()

### 3.3 Gráfico 3: Arquitectura Genómica de la Isla Catabólica de Degradación de PET

Al analizar la distribución física en el cromosoma de los genes encontrados (`EGEIPIOO_03514` a `EGEIPIOO_03530`), descubrimos un hallazgo evolutivo fascinante: **todos los genes requeridos para degradar el PET y su monómero TPA están agrupados en una sola isla genómica catabólica**, flanqueada por transposasas de la familia **IS91**.

Esto sugiere fuertemente que la bacteria adquirió o reorganizó recientemente esta maquinaria catabólica por transferencia horizontal de genes (HGT) bajo la presión selectiva de vivir en un ambiente contaminado con plásticos.

Grafiquemos el mapa físico del clúster génico:

In [ ]:
# Clúster genómico entre EGEIPIOO_03514 y EGEIPIOO_03530
cluster_tags = [f"EGEIPIOO_035{i}" for i in range(14, 31)]
df_cluster = df_prokka[df_prokka["locus_tag"].isin(cluster_tags)].copy()

gene_labels = {
    "EGEIPIOO_03514": "pcaR (Regulador)",
    "EGEIPIOO_03515": "MHETasa (3.1.1.102)",
    "EGEIPIOO_03518": "tphA2 (TPA dioxigenasa α)",
    "EGEIPIOO_03519": "tphA3 (TPA dioxigenasa β)",
    "EGEIPIOO_03520": "tphB (Dihidrodiol DH)",
    "EGEIPIOO_03521": "tphA1 (TPA reductasa)",
    "EGEIPIOO_03522": "hcaR (Activador)",
    "EGEIPIOO_03524": "IS91 (Transposasa)",
    "EGEIPIOO_03526": "pyrC (Dihidroorotasa)",
    "EGEIPIOO_03527": "PETasa (3.1.1.101)",
    "EGEIPIOO_03528": "lifO (Chaperona lipasa)",
    "EGEIPIOO_03530": "IS91 (Transposasa)"
}

color_map = {
    "MHETasa (3.1.1.102)": "#e41a1c",
    "PETasa (3.1.1.101)": "#e41a1c",
    "tphA2 (TPA dioxigenasa α)": "#377eb8",
    "tphA3 (TPA dioxigenasa β)": "#377eb8",
    "tphB (Dihidrodiol DH)": "#377eb8",
    "tphA1 (TPA reductasa)": "#377eb8",
    "IS91 (Transposasa)": "#984ea3",
    "pcaR (Regulador)": "#ff7f00",
    "hcaR (Activador)": "#ff7f00",
    "lifO (Chaperona lipasa)": "#4daf4a",
    "pyrC (Dihidroorotasa)": "#999999",
    "Hipotética / Otro": "#e0e0e0"
}

fig, ax = plt.subplots(figsize=(14, 4))

y_pos = 0.5
for idx, row in df_cluster.reset_index(drop=True).iterrows():
    tag = row["locus_tag"]
    label = gene_labels.get(tag, "Hipotética / Otro")
    color = color_map.get(label, "#e0e0e0")
    
    # Rectángulo que representa el gen
    rect = plt.Rectangle((idx * 1.5, y_pos - 0.2), 1.2, 0.4, color=color, edgecolor="#222222", linewidth=1)
    ax.add_patch(rect)
    
    # Etiquetas del gen
    if label != "Hipotética / Otro":
        ax.text(idx * 1.5 + 0.6, y_pos + 0.35, label, rotation=45, ha="left", va="bottom", fontsize=9, fontweight="bold")
    else:
        ax.text(idx * 1.5 + 0.6, y_pos - 0.35, "hip.", ha="center", va="top", fontsize=8, color="#666666")

ax.set_xlim(-1, len(df_cluster) * 1.5 + 2)
ax.set_ylim(0, 1.4)
ax.axis("off")
plt.title("Arquitectura de la Isla Genómica de Asimilación de PET en Ideonella sakaiensis\n(PETasa + Chaperona LifO + Transposasas IS91 + Operón TPA + MHETasa + Regulador PcaR)", fontsize=12, fontweight="bold", pad=20)
plt.tight_layout()
plt.show()

## 4. Control de Calidad Biológica con BUSCO ("Evaluamos Juntos")

Una pregunta crítica en cualquier proyecto genómico es: *¿Podemos confiar en que el pipeline predijo de manera completa los genes del organismo sin omitir rutas esenciales ni truncar proteínas?*

A diferencia de QUAST (que evalúa métricas físicas y estructurales de continuidad como N50 y L50), **BUSCO** (*Benchmarking Universal Single-Copy Orthologs*) evalúa la **calidad biológica** buscando genes evolutivamente obligatorios y conservados en copia única dentro de un linaje taxonómico.

En esta ocasión, evaluaremos el archivo proteico traducido generado por Prokka (`resultados/prokka_ideonella/ideonella_sakaiensis.faa`) usando el modo proteínas (`-m proteins`) y el conjunto de datos `bacteria_odb10`:

In [ ]:
# =========================================================================
# PARÁMETROS CONFIGURABLES: Evaluación BUSCO del Proteoma
# =========================================================================
PROKKA_FAA = f"{OUT_PROKKA}/ideonella_sakaiensis.faa"
OUT_BUSCO = "resultados/busco_ideonella"

cmd_busco = f"busco -i {PROKKA_FAA} -m proteins -l bacteria_odb10 -o {OUT_BUSCO} -f -c {HILOS}"
print(f"Ejecutando BUSCO:\n{cmd_busco}\n")

if not glob.glob(f"{OUT_BUSCO}/*.json"):
    res_busco = subprocess.run(cmd_busco, shell=True, capture_output=True, text=True)
    print(res_busco.stdout if res_busco.stdout else res_busco.stderr)
else:
    print(" [INFO] Los resultados de BUSCO ya existen en resultados/busco_ideonella. ¡Cargando resultados!")

### 4.2 Gráfico 4: Evaluación de Completitud de BUSCO (Barras Apiladas)

Las categorías estandarizadas de BUSCO son:
* **Complete & Single-copy (S):** Genes ortólogos encontrados completos y en una sola copia esperada.
* **Complete & Duplicated (D):** Genes encontrados completos pero en múltiples copias (posibles duplicaciones o parálogos).
* **Fragmented (F):** Genes encontrados parcialmente (indicativo de proteínas truncadas o errores de ensamblaje en los extremos del contig).
* **Missing (M):** Genes ausentes no detectados en la anotación.

Leamos el resumen estructurado y grafiquemos el resultado con la paleta oficial de BUSCO:

In [ ]:
def parse_busco(busco_dir):
    json_files = glob.glob(f"{busco_dir}/*.json")
    if json_files:
        with open(json_files[0]) as f:
            data = json.load(f)
        res = data.get("results", {})
        s = res.get("Single copy percentage", res.get("Single percentage", 0.0))
        d = res.get("Multi copy percentage", res.get("Multi percentage", 0.0))
        f_val = res.get("Fragmented percentage", 0.0)
        m = res.get("Missing percentage", 0.0)
        c = res.get("Complete percentage", s + d)
        n = res.get("n_markers", res.get("Total buscos searched", 0))
        return {"Complete": c, "Complete_Single": s, "Complete_Duplicated": d, "Fragmented": f_val, "Missing": m, "Total": n}
    
    txt_files = glob.glob(f"{busco_dir}/short_summary.*.txt")
    if txt_files:
        with open(txt_files[0]) as f:
            text = f.read()
        s = float(re.search(r"S:([0-9.]+)%", text).group(1))
        d = float(re.search(r"D:([0-9.]+)%", text).group(1))
        f_val = float(re.search(r"F:([0-9.]+)%", text).group(1))
        m = float(re.search(r"M:([0-9.]+)%", text).group(1))
        c = float(re.search(r"C:([0-9.]+)%", text).group(1))
        return {"Complete": c, "Complete_Single": s, "Complete_Duplicated": d, "Fragmented": f_val, "Missing": m, "Total": 124}
    
    raise FileNotFoundError(f"No se encontró resumen de BUSCO en {busco_dir}")

busco_stats = parse_busco(OUT_BUSCO)

plt.figure(figsize=(10, 2.8))

s = busco_stats["Complete_Single"]
d = busco_stats["Complete_Duplicated"]
f = busco_stats["Fragmented"]
m = busco_stats["Missing"]

# Paleta oficial de BUSCO
c_single = "#3498db"     # Azul
c_dup = "#2ecc71"        # Verde
c_frag = "#f39c12"       # Amarillo/Naranja
c_miss = "#e74c3c"       # Rojo

plt.barh(["I. sakaiensis (Proteoma)"], [s], color=c_single, label=f"Completo y copia única (S): {s:.1f}%")
plt.barh(["I. sakaiensis (Proteoma)"], [d], left=[s], color=c_dup, label=f"Completo y duplicado (D): {d:.1f}%")
plt.barh(["I. sakaiensis (Proteoma)"], [f], left=[s+d], color=c_frag, label=f"Fragmentado (F): {f:.1f}%")
plt.barh(["I. sakaiensis (Proteoma)"], [m], left=[s+d+f], color=c_miss, label=f"Ausente (M): {m:.1f}%")

plt.xlim(0, 100)
plt.xlabel("% de genes BUSCO (bacteria_odb10, n=124)", fontsize=11)
plt.title(f"Evaluación de Calidad Biológica del Proteoma Anotado con BUSCO\nCompletitud Total: {s + d:.1f}%", fontsize=12, fontweight="bold", pad=12)
plt.legend(bbox_to_anchor=(0.5, -0.38), loc="upper center", ncol=4, frameon=True, facecolor="white", edgecolor="#cccccc", fontsize=9)
plt.tight_layout()
plt.show()

---
## 5. Preguntas Formativas y Pensamiento Crítico

Responde a las siguientes preguntas justificando tus respuestas con los resultados y gráficos obtenidos en este taller:

### Pregunta 1: Eficacia de Prokka vs. eggNOG-mapper
> Al revisar la salida inicial de Prokka, encontramos que aproximadamente la mitad de los genes fueron catalogados como *"hypothetical protein"*.
> **¿A qué se debe esta limitación en Prokka y cómo la estrategia de agrupamiento por ortología no supervisada de eggNOG-mapper logra rescatar la función biológica de muchos de estos genes sin incurrir en transferencias erróneas por parálogos?**

> ✍️ **Respuesta:** [Escribe tu análisis aquí]

### Pregunta 2: Arquitectura y Compartimentalización de la Ruta del PET
> En el análisis de ontología celular (Cellular Component), observamos que la **PETasa** se asigna a la *región extracelular* (`GO:0005576`), mientras que la **MHETasa** se localiza en la *membrana celular externa* (`GO:0009279`).
> **¿Por qué esta compartimentalización representa una ventaja termodinámica y evolutiva crucial para la bacteria? ¿Qué sucedería si la bacteria intentara internalizar el polímero de PET directamente al citoplasma sin estas enzimas externas?**

> ✍️ **Respuesta:** [Escribe tu análisis aquí]

### Pregunta 3: Interpretación de la Completitud con BUSCO
> El análisis de BUSCO arrojó una completitud del **98.4%** ($S=96.8\%$, $D=1.6\%$, $F=0.8\%$, $M=0.8\%$).
> **¿Qué conclusiones podemos inferir sobre la calidad de la predicción de marcos de lectura de Prodigal y la ausencia de errores de fase (frameshifts) o quimeras en la secuencia ensamblada?**

> ✍️ **Respuesta:** [Escribe tu análisis aquí]

### Pregunta 4: Desafíos de Ingeniería en Biorreactores de Reciclaje Enzimático
> En la industria biotecnológica, varios laboratorios intentaron producir únicamente la enzima PETasa recombinante en *E. coli*. Sin embargo, descubrieron que la reacción se desacelera rápidamente debido a una fuerte **inhibición competitiva por producto causada por la acumulación de MHET**.
> **A la luz de la isla genómica que descubriste en *Ideonella sakaiensis* (Gráfico 3), ¿cómo resuelve la bacteria este cuello de botella y por qué un diseño de bioproceso con co-inmovilización enzimática (PETasa + MHETasa) o un consorcio microbiano completo es superior a emplear la PETasa aislada?**

> ✍️ **Respuesta:** [Escribe tu análisis aquí]

---
*Pontificia Universidad Javeriana — Bioinformática para Ingenieros*  
*Elaborado por: Juan Sebastián Malagón Torres, Eliana Zuluaga Aponte*
